In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv('Credit Risk.csv')
df.head()

,Loan_ID,Gender,Married,Dependents,Education,Self_Employed,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History,Property_Area,Loan_Status
0,LP001002,Male,No,0,Graduate,No,5849,0.0,NaN,360.0,1.0,Urban,Y
1,LP001003,Male,Yes,1,Graduate,No,4583,1508.0,128.0,360.0,1.0,Rural,N
2,LP001005,Male,Yes,0,Graduate,Yes,3000,0.0,66.0,360.0,1.0,Urban,Y
3,LP001006,Male,Yes,0,Not Graduate,No,2583,2358.0,120.0,360.0,1.0,Urban,Y
4,LP001008,Male,No,0,Graduate,No,6000,0.0,141.0,360.0,1.0,Urban,Y


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 614 entries, 0 to 613
Data columns (total 13 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Loan_ID            614 non-null    object 
 1   Gender             601 non-null    object 
 2   Married            611 non-null    object 
 3   Dependents         599 non-null    object 
 4   Education          614 non-null    object 
 5   Self_Employed      582 non-null    object 
 6   ApplicantIncome    614 non-null    int64  
 7   CoapplicantIncome  614 non-null    float64
 8   LoanAmount         592 non-null    float64
 9   Loan_Amount_Term   600 non-null    float64
 10  Credit_History     564 non-null    float64
 11  Property_Area      614 non-null    object 
 12  Loan_Status        614 non-null    object 
dtypes: float64(4), int64(1), object(8)
memory usage: 62.5+ KB


In [4]:
df.describe()

,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History
count,614.000000,614.000000,592.000000,600.00000,564.000000
mean,5403.459283,1621.245798,146.412162,342.00000,0.842199
std,6109.041673,2926.248369,85.587325,65.12041,0.364878
min,150.000000,0.000000,9.000000,12.00000,0.000000
25%,2877.500000,0.000000,100.000000,360.00000,1.000000
50%,3812.500000,1188.500000,128.000000,360.00000,1.000000
75%,5795.000000,2297.250000,168.000000,360.00000,1.000000
max,81000.000000,41667.000000,700.000000,480.00000,1.000000


##### Прибираю дивне значення '3+', замінюю просто на 3

In [5]:
df['Dependents'].value_counts()

Dependents
0     345
1     102
2     101
3+     51
Name: count, dtype: int64

In [6]:
df['Dependents'] = df['Dependents'].replace('3+', 3)
df['Dependents'].value_counts()

Dependents
0    345
1    102
2    101
3     51
Name: count, dtype: int64

##### Розділяю категоріальні та числові ознаки, і додаю колонку для числового таргету.
##### Ознаку Loan_ID дропну бо нема в ній сенсу. Це порядкове значення в списку, не впливає на таргет.

In [7]:
df['Loan_Status'].value_counts()

Loan_Status
Y    422
N    192
Name: count, dtype: int64

In [8]:
df['Loan_Status_enc'] = df['Loan_Status'].map({'Y': 1, 'N': 0})
df.head()

,Loan_ID,Gender,Married,Dependents,Education,Self_Employed,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History,Property_Area,Loan_Status,Loan_Status_enc
0,LP001002,Male,No,0,Graduate,No,5849,0.0,NaN,360.0,1.0,Urban,Y,1
1,LP001003,Male,Yes,1,Graduate,No,4583,1508.0,128.0,360.0,1.0,Rural,N,0
2,LP001005,Male,Yes,0,Graduate,Yes,3000,0.0,66.0,360.0,1.0,Urban,Y,1
3,LP001006,Male,Yes,0,Not Graduate,No,2583,2358.0,120.0,360.0,1.0,Urban,Y,1
4,LP001008,Male,No,0,Graduate,No,6000,0.0,141.0,360.0,1.0,Urban,Y,1


In [9]:
cat_features = ['Gender', 'Married', 'Education', 'Self_Employed', 'Property_Area']
num_features = ['Dependents', 'ApplicantIncome', 'CoapplicantIncome', 'LoanAmount', 'Loan_Amount_Term', 'Credit_History']

In [10]:
X = df[cat_features + num_features]
y = df['Loan_Status_enc']

In [11]:
from sklearn.model_selection import train_test_split

In [12]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=15)
X_train.shape, X_test.shape, y_train.shape, y_test.shape

((429, 11), (185, 11), (429,), (185,))

##### Спробуємо LogisticRegression

In [1]:
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer

In [14]:
from sklearn.metrics import accuracy_score
from sklearn.model_selection import GridSearchCV

In [15]:
num_pipe = Pipeline([
    ('imputer', SimpleImputer()),
    ('scaler', StandardScaler())
])

cat_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encodind', OneHotEncoder(sparse_output=False, handle_unknown='ignore', drop="if_binary"))
])

preprocessor = ColumnTransformer([
    ('num', num_pipe, num_features),
    ('cat', cat_pipe, cat_features)
])

lr_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', LogisticRegression(class_weight='balanced'))
])

In [16]:
param_grid = {
    'model__solver': ['liblinear','saga'],
    'model__penalty': ['l1', 'l2'],
    'model__C': [0.01, 0.1, 1, 10, 100],
    'preprocessor__num__imputer__strategy': ['mean', 'median']
}

In [17]:
grid_search_lr = GridSearchCV(
    lr_pipeline,
    param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1
)

In [18]:
grid_search_lr.fit(X_train, y_train)

,estimator,Pipeline(step...'balanced'))])
,param_grid,"{'model__C': [0.01, 0.1, ...], 'model__penalty': ['l1', 'l2'], 'model__solver': ['liblinear', 'saga'], 'preprocessor__num__imputer__strategy': ['mean', 'median']}"
,scoring,'accuracy'
,n_jobs,-1
,refit,True
,cv,5
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,transformers,"[('num', ...), ('cat', ...)]"


In [19]:
y_pred_lr = grid_search_lr.predict(X_test)

In [20]:
from sklearn.metrics import classification_report

In [21]:
print(classification_report(y_test, y_pred_lr))

              precision    recall  f1-score   support

           0       0.94      0.47      0.63        66
           1       0.77      0.98      0.86       119

    accuracy                           0.80       185
   macro avg       0.85      0.73      0.74       185
weighted avg       0.83      0.80      0.78       185



##### Бачимо, що для класу "0" (відмовити), модель точна тільки на 47%, погана тенденція для банку так витрачати гроші.
##### Спробуємо іншу модель навчання Naive Bayes.

In [22]:
from sklearn.naive_bayes import GaussianNB

In [23]:
num_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

cat_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encodind', OneHotEncoder(sparse_output=False, handle_unknown='ignore', drop="if_binary"))
])

preprocessor = ColumnTransformer([
    ('num', num_pipe, num_features),
    ('cat', cat_pipe, cat_features)
])

gnb_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', GaussianNB())
])

In [24]:
gnb_pipeline.fit(X_train, y_train)

,steps,"[('preprocessor', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [25]:
y_pred_gnb = gnb_pipeline.predict(X_test)

In [26]:
print(classification_report(y_test, y_pred_gnb))

              precision    recall  f1-score   support

           0       0.94      0.48      0.64        66
           1       0.77      0.98      0.87       119

    accuracy                           0.81       185
   macro avg       0.86      0.73      0.75       185
weighted avg       0.83      0.81      0.79       185



##### На 1% піднялися. Йдемо до іншої моделі - Decision tree.

In [27]:
from sklearn.tree import DecisionTreeClassifier

In [28]:
num_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

cat_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encodind', OneHotEncoder(sparse_output=False, handle_unknown='ignore', drop="if_binary"))
])

preprocessor = ColumnTransformer([
    ('num', num_pipe, num_features),
    ('cat', cat_pipe, cat_features)
])

dtc_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', DecisionTreeClassifier(class_weight='balanced', random_state=15))
])

In [29]:
param_grid = {
    'model__criterion': ['gini', 'entropy', 'log_loss'],
    'model__max_depth': [3, 5, 10, 15, 20, None]
}

In [30]:
grid_search_dtc = GridSearchCV(
    dtc_pipeline,
    param_grid,
    cv=5,
    scoring='f1',
    n_jobs=-1
)

In [31]:
grid_search_dtc.fit(X_train, y_train)

,estimator,Pipeline(step...m_state=15))])
,param_grid,"{'model__criterion': ['gini', 'entropy', ...], 'model__max_depth': [3, 5, ...]}"
,scoring,'f1'
,n_jobs,-1
,refit,True
,cv,5
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,transformers,"[('num', ...), ('cat', ...)]"


In [32]:
y_pred_dtc = grid_search_dtc.predict(X_test)

In [33]:
print(classification_report(y_test, y_pred_dtc))

              precision    recall  f1-score   support

           0       0.65      0.50      0.56        66
           1       0.75      0.85      0.80       119

    accuracy                           0.72       185
   macro avg       0.70      0.67      0.68       185
weighted avg       0.72      0.72      0.71       185



##### Для scoring використала цього разу F1 замість accuracy. Підняла трохи recall але втратила в accuracy.

In [34]:
from sklearn.neighbors import KNeighborsClassifier

In [35]:
num_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

cat_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encodind', OneHotEncoder(sparse_output=False, handle_unknown='ignore', drop="if_binary"))
])

preprocessor = ColumnTransformer([
    ('num', num_pipe, num_features),
    ('cat', cat_pipe, cat_features)
])

knn_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', KNeighborsClassifier())
])

In [36]:
param_grid = {
    'model__n_neighbors': [3, 5, 7, 9, 11, 15, 21]
}

In [37]:
grid_search_knn = GridSearchCV(
    knn_pipeline,
    param_grid,
    cv=5,
    scoring='f1',
    n_jobs=-1
)

In [38]:
grid_search_knn.fit(X_train, y_train)

,estimator,Pipeline(step...lassifier())])
,param_grid,"{'model__n_neighbors': [3, 5, ...]}"
,scoring,'f1'
,n_jobs,-1
,refit,True
,cv,5
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,transformers,"[('num', ...), ('cat', ...)]"


In [39]:
y_pred_knn = grid_search_knn.predict(X_test)

In [40]:
print(classification_report(y_test, y_pred_knn))

              precision    recall  f1-score   support

           0       0.94      0.47      0.63        66
           1       0.77      0.98      0.86       119

    accuracy                           0.80       185
   macro avg       0.85      0.73      0.74       185
weighted avg       0.83      0.80      0.78       185



##### Наче маємо непогіна показники хоч recall для відмови в кредиті все одно низький.
##### Спробуємо прибрати викиди.

In [41]:
df.describe()

,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History,Loan_Status_enc
count,614.000000,614.000000,592.000000,600.00000,564.000000,614.000000
mean,5403.459283,1621.245798,146.412162,342.00000,0.842199,0.687296
std,6109.041673,2926.248369,85.587325,65.12041,0.364878,0.463973
min,150.000000,0.000000,9.000000,12.00000,0.000000,0.000000
25%,2877.500000,0.000000,100.000000,360.00000,1.000000,0.000000
50%,3812.500000,1188.500000,128.000000,360.00000,1.000000,1.000000
75%,5795.000000,2297.250000,168.000000,360.00000,1.000000,1.000000
max,81000.000000,41667.000000,700.000000,480.00000,1.000000,1.000000


In [42]:
Q1 = df['ApplicantIncome'].quantile(0.25)
Q3 = df['ApplicantIncome'].quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

df_clean = df[(df['ApplicantIncome'] >= lower_bound) & (df['ApplicantIncome'] <= upper_bound)]

In [43]:
df_clean.info()

<class 'pandas.core.frame.DataFrame'>
Index: 564 entries, 0 to 613
Data columns (total 14 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Loan_ID            564 non-null    object 
 1   Gender             554 non-null    object 
 2   Married            561 non-null    object 
 3   Dependents         550 non-null    object 
 4   Education          564 non-null    object 
 5   Self_Employed      534 non-null    object 
 6   ApplicantIncome    564 non-null    int64  
 7   CoapplicantIncome  564 non-null    float64
 8   LoanAmount         544 non-null    float64
 9   Loan_Amount_Term   550 non-null    float64
 10  Credit_History     517 non-null    float64
 11  Property_Area      564 non-null    object 
 12  Loan_Status        564 non-null    object 
 13  Loan_Status_enc    564 non-null    int64  
dtypes: float64(4), int64(2), object(8)
memory usage: 66.1+ KB


##### Спробуємо додати нові колонки - загальний дохід усіх членів родини (TotalIncome), виплати по кредиту щомісяця (LoanPerMonth)

In [51]:
df_clean['TotalIncome'] = df_clean['ApplicantIncome'] + df_clean['CoapplicantIncome']
df_clean['LoanPerMonth'] = df_clean['LoanAmount'] / df_clean['Loan_Amount_Term']
df_clean.info()

<class 'pandas.core.frame.DataFrame'>
Index: 564 entries, 0 to 613
Data columns (total 16 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Loan_ID            564 non-null    object 
 1   Gender             554 non-null    object 
 2   Married            561 non-null    object 
 3   Dependents         550 non-null    object 
 4   Education          564 non-null    object 
 5   Self_Employed      534 non-null    object 
 6   ApplicantIncome    564 non-null    int64  
 7   CoapplicantIncome  564 non-null    float64
 8   LoanAmount         544 non-null    float64
 9   Loan_Amount_Term   550 non-null    float64
 10  Credit_History     517 non-null    float64
 11  Property_Area      564 non-null    object 
 12  Loan_Status        564 non-null    object 
 13  Loan_Status_enc    564 non-null    int64  
 14  TotalIncome        564 non-null    float64
 15  LoanPerMonth       530 non-null    float64
dtypes: float64(6), int64(2), object

C:\Users\dell\AppData\Local\Temp\ipykernel_19772\2539717437.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_clean['TotalIncome'] = df_clean['ApplicantIncome'] + df_clean['CoapplicantIncome']
C:\Users\dell\AppData\Local\Temp\ipykernel_19772\2539717437.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_clean['LoanPerMonth'] = df_clean['LoanAmount'] / df_clean['Loan_Amount_Term']


In [55]:
X_cl = df_clean.drop(['Loan_Status_enc'], axis=1)
y_cl = df_clean['Loan_Status_enc']
X_cl.shape, y_cl.shape

((564, 15), (564,))

##### Трошки прибрала людей з великим доходом. Запущу зараз ще раз навчання на новому датасеті.

In [56]:
grid_search_lr.fit(X_cl, y_cl)

,estimator,Pipeline(step...'balanced'))])
,param_grid,"{'model__C': [0.01, 0.1, ...], 'model__penalty': ['l1', 'l2'], 'model__solver': ['liblinear', 'saga'], 'preprocessor__num__imputer__strategy': ['mean', 'median']}"
,scoring,'accuracy'
,n_jobs,-1
,refit,True
,cv,5
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,transformers,"[('num', ...), ('cat', ...)]"


In [57]:
y_pred_lr_cl = grid_search_lr.predict(X_test)

In [58]:
print(classification_report(y_test, y_pred_lr_cl))

              precision    recall  f1-score   support

           0       0.94      0.47      0.63        66
           1       0.77      0.98      0.86       119

    accuracy                           0.80       185
   macro avg       0.85      0.73      0.74       185
weighted avg       0.83      0.80      0.78       185



In [59]:
grid_search_dtc.fit(X_cl, y_cl)

,estimator,Pipeline(step...m_state=15))])
,param_grid,"{'model__criterion': ['gini', 'entropy', ...], 'model__max_depth': [3, 5, ...]}"
,scoring,'f1'
,n_jobs,-1
,refit,True
,cv,5
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,transformers,"[('num', ...), ('cat', ...)]"


In [60]:
y_pred_dtc_cl = grid_search_dtc.predict(X_test)

In [61]:
print(classification_report(y_test, y_pred_dtc_cl))

              precision    recall  f1-score   support

           0       0.97      0.48      0.65        66
           1       0.78      0.99      0.87       119

    accuracy                           0.81       185
   macro avg       0.87      0.74      0.76       185
weighted avg       0.85      0.81      0.79       185



In [62]:
grid_search_knn.fit(X_cl, y_cl)

,estimator,Pipeline(step...lassifier())])
,param_grid,"{'model__n_neighbors': [3, 5, ...]}"
,scoring,'f1'
,n_jobs,-1
,refit,True
,cv,5
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,transformers,"[('num', ...), ('cat', ...)]"


In [63]:
y_pred_knn_cl = grid_search_knn.predict(X_test)

In [64]:
print(classification_report(y_test, y_pred_knn_cl))

              precision    recall  f1-score   support

           0       0.94      0.45      0.61        66
           1       0.76      0.98      0.86       119

    accuracy                           0.79       185
   macro avg       0.85      0.72      0.74       185
weighted avg       0.83      0.79      0.77       185



In [65]:
gnb_pipeline.fit(X_cl, y_cl)

,steps,"[('preprocessor', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [66]:
y_pred_gnb_cl = gnb_pipeline.predict(X_test)

In [67]:
print(classification_report(y_test, y_pred_gnb_cl))

              precision    recall  f1-score   support

           0       0.94      0.50      0.65        66
           1       0.78      0.98      0.87       119

    accuracy                           0.81       185
   macro avg       0.86      0.74      0.76       185
weighted avg       0.84      0.81      0.79       185



#### Висновки
##### На даному датасеті найкраще спрацював Naive Bayes, хоча суттєво підняти recall не вийшло.
##### Для роботи з даними була спроба викинути викиди по доходах та додані нові ознаки. Ці дії нічого не покращили.